# 🚀 Servidor OpenCode en Google Colab con CCUs (Google AI Pro)

### ⚠️ PASO 1 PREVIO OBLIGATORIO (Para usar tus CCUs de Google AI Pro con GPU):
Andá arriba al menú: **Entorno de ejecución > Cambiar tipo de entorno de ejecución** y seleccioná **GPU** (T4, L4 o A100). Hacé clic en **Guardar**.

---

### ⚡ PASO 2: Dale PLAY a la celda de abajo
Esta única celda hace todo automáticamente:
1. Instala Ollama y Cloudflare Tunnel.
2. Descarga **Qwen 2.5 Coder 7B** a la GPU.
3. Levanta el túnel seguro y te imprime el link en verde.

In [ ]:
# 🚀 TODO-EN-UNO: Instalar, descargar Qwen 2.5 Coder e iniciar Túnel Cloudflare
import os, sys, time, re, subprocess

print("==================================================================")
print("1/4 📦 Instalando Ollama en Google Colab...")
print("==================================================================")
!curl -fsSL https://ollama.ai/install.sh | sh

print("\n==================================================================")
print("2/4 🌐 Descargando Cloudflare Tunnel...")
print("==================================================================")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

print("\n==================================================================")
print("3/4 ⚡ Iniciando Ollama y descargando Qwen 2.5 Coder...")
print("==================================================================")
!pkill -f "ollama serve" || true
!pkill -f "cloudflared" || true
time.sleep(2)

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
subprocess.Popen(["ollama", "serve"])
time.sleep(4)

# Descargar Qwen 2.5 Coder 7B
!ollama pull qwen2.5-coder:7b

print("\n==================================================================")
print("4/4 🚀 Levantando Túnel Cloudflare seguro...")
print("==================================================================")
tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()

while time.time() - start_time < 40:
    line = tunnel.stdout.readline()
    if not line: continue
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*65)
    print("🟢 ¡TU SERVIDOR OPENCODE ESTÁ EN VIVO EN LA NUBE DE GOOGLE!")
    print("="*65)
    print(f"\n👉 URL DEL TÚNEL PARA COPIAR:")
    print(f"   {public_url}\n")
    print(f"👉 MODELO ACTIVO: qwen2.5-coder:7b")
    print("="*65)
    print("\n📋 CÓMO VINCULARLO CON ANTIGRAVITY:")
    print("Pegá esa URL en el chat o escribí:")
    print(f'   "Conectate a {public_url}"')
    print("="*65 + "\n")
    
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("Túnel detenido.")
else:
    print("⚠️ Error obteniendo URL del túnel. Registros:")
    print(tunnel.communicate()[0])